# PasteTrace — Mamba Behavioral Sequence Model

**Yêu cầu:** Vào `Runtime → Change runtime type → T4 GPU` trước khi chạy.

Dataset: **205 sinh viên tổng hợp** trong `test_new_cohort/` (đã có sẵn trong repo — không cần upload Drive).

## Pipeline
```
Bước 0  Setup   (kiểm tra GPU, cài thư viện, clone repo)
Bước 1  Build   (meta.json → chuỗi sự kiện, data/train_sequences/)
Bước 2  Train   (Mamba model, lưu models/mamba/mamba.pt)
Bước 3  Lưu     (download mamba.pt về máy hoặc lên Drive)
Bước 4  Predict (chạy inference trên thư mục tests/)
```

## Bước 0a — Kiểm tra GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️ GPU not enabled. Enable in Notebook Settings → GPU')

## Bước 0b — Cài thư viện Mamba (CHIA 2 CELL — kernel restart giữa chừng)

**Cell 4**: Pin torch 2.7.1+cu126 rồi `os._exit(0)` để restart kernel.
**Cell 4b**: Verify torch + build causal-conv1d + mamba-ssm từ source.

Lý do phải restart kernel: C extensions đã load vào memory, `del sys.modules` không thể reload.
Nếu skip bước này sẽ gặp `function '_has_torch_function' already has a docstring`.


In [ ]:
# IMPORTANT: Kaggle default image ships torch 2.x with cu130 (CUDA 13.0).
# We MUST pin torch 2.7.1+cu126 BEFORE building causal-conv1d/mamba-ssm,
# otherwise pip dependency resolver will pull torch 2.10/2.13 (cu128/cu130)
# → ABI mismatch → CUDA_VERSION_MISMATCH when building extensions.
#
# Use DIRECT WHEEL URLs (not --index-url) because Kaggle's pip cache has been
# observed to ignore --index-url and resolve from PyPI default (latest torch).
import os, subprocess, sys

def run(cmd, desc, timeout=600):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    ok = r.returncode == 0
    tag = '[OK]  ' if ok else '[FAIL]'
    print(f'  {tag} {desc}')
    if not ok:
        print(f'        stderr: {r.stderr[-400:]}')
    return ok

TORCH_CUDA = 'cu126'
TORCH_VERSION = '2.7.1'
PY_TAG = 'cp312-cp312'
PLAT_TAG = 'manylinux_2_28_x86_64'
BASE = f'https://download.pytorch.org/whl/{TORCH_CUDA}'
URLS = {
    'torch':       f'{BASE}/torch-{TORCH_VERSION}%2B{TORCH_CUDA}-{PY_TAG}-{PLAT_TAG}.whl',
    'torchvision': f'{BASE}/torchvision-0.22.1%2B{TORCH_CUDA}-{PY_TAG}-{PLAT_TAG}.whl',
    'torchaudio':  f'{BASE}/torchaudio-{TORCH_VERSION}%2B{TORCH_CUDA}-{PY_TAG}-{PLAT_TAG}.whl',
}

print('=' * 60)
print(f'Pinning torch {TORCH_VERSION}+{TORCH_CUDA} (direct wheel URL)')
print('=' * 60)
run(['pip', 'uninstall', '-q', '-y', 'torch', 'torchvision', 'torchaudio'],
    'uninstall old torch')
for pkg, url in URLS.items():
    run(['pip', 'install', '-q', '--no-deps', url], f'install {pkg} {url.split("/")[-1]}')

# QUAN TRỌNG: --no-deps đã bỏ luôn các NVIDIA CUDA runtime libs (cusparseLt, cuBLAS, cuDNN, NCCL...)
# → kernel mới sau restart sẽ fail `libcusparseLt.so.0` nếu không cài lại.
# Cài NVIDIA deps TRƯỚC SystemExit, để kernel restart thấy .so ngay khi load torch.
_nvidia_pkgs = [
    'nvidia-cuda-nvrtc-cu12==12.6.77',
    'nvidia-cuda-runtime-cu12==12.6.77',
    'nvidia-cuda-cupti-cu12==12.6.80',
    'nvidia-cudnn-cu12==9.5.1.17',
    'nvidia-cublas-cu12==12.6.4.1',
    'nvidia-cufft-cu12==11.3.0.4',
    'nvidia-curand-cu12==10.3.7.77',
    'nvidia-cusolver-cu12==11.7.1.2',
    'nvidia-cusparse-cu12==12.5.4.2',
    'nvidia-cusparselt-cu12==0.6.3',
    'nvidia-nccl-cu12==2.26.2',
    'nvidia-nvtx-cu12==12.6.77',
    'nvidia-nvjitlink-cu12==12.6.85',
    'nvidia-cufile-cu12==1.11.1.6',
    'triton==3.3.1',
]
run(['pip', 'install', '-q', '--no-deps'] + _nvidia_pkgs,
    f'nvidia-cu12 runtimes + triton {len(_nvidia_pkgs)} pkgs (BEFORE restart)')

# Verify bằng subprocess MỚI — load lại .so từ đầu, chứng minh kernel mới sẽ OK.
_v = subprocess.run(
    [sys.executable, '-c',
     'import torch; x = torch.zeros(2,2, device="cuda"); print("SMOKE:", x)'],
    capture_output=True, text=True, timeout=60)
if _v.returncode == 0:
    print(f'  [OK] fresh-process smoke: {_v.stdout.strip()}')
else:
    print(f'  [FAIL] fresh-process smoke:\n{_v.stderr[-400:]}')
    raise RuntimeError('Torch+CUDA libs still missing — check pip install output above')

print()
print('=' * 60)
print('>>> MUST RESTART JUPYTER KERNEL NOW <<<')
print('=' * 60)
print('C extensions loaded into memory cannot be reloaded via importlib.')
print('Run this in a NEW cell:')
print('    import os; os._exit(0)')
print('After kernel restarts, continue with Cell 4b below.')
raise SystemExit(0)


In [ ]:
# RUN THIS AFTER RESTARTING KERNEL (after Cell 4 os._exit)
import os, subprocess, sys, torch

# Pin torch wheel URLs (same as Cell 4 — phải redefine sau khi kernel restart)
URLS = {
    'torch':       'https://download.pytorch.org/whl/cu126/torch-2.7.1%2Bcu126-cp312-cp312-manylinux_2_28_x86_64.whl',
    'torchvision': 'https://download.pytorch.org/whl/cu126/torchvision-0.22.1%2Bcu126-cp312-cp312-manylinux_2_28_x86_64.whl',
    'torchaudio':  'https://download.pytorch.org/whl/cu126/torchaudio-2.7.1%2Bcu126-cp312-cp312-manylinux_2_28_x86_64.whl',
}

def run(cmd, desc, timeout=600):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    ok = r.returncode == 0
    tag = '[OK]  ' if ok else '[FAIL]'
    print(f'  {tag} {desc}')
    if not ok:
        print(f'        stderr: {r.stderr[-400:]}')
    return ok

print(f'Torch __version__: {torch.__version__}')
print(f'Torch CUDA        : {torch.version.cuda}')
# Phát hiện nếu kernel CHƯA restart: torch.__file__ vẫn trỏ đến .dist-packages,
# nhưng torch.version đã khác với wheel vừa install → conflict.
print(f'Torch location    : {torch.__file__}')
assert torch.__version__.startswith('2.7.'), (
    f'Expected 2.7.x, got {torch.__version__}. \n'
    'Did you forget to RESTART the kernel after Cell 4? '
    'Run `import os; os._exit(0)` then re-run this cell.')
assert torch.version.cuda in ('12.6', '12.8'), f'Unexpected CUDA: {torch.version.cuda}'
print('[OK] torch 2.7.1+cu126 confirmed\n')

# Smoke test trên kernel hiện tại (đã restart) — verify CUDA init + GPU tensor end-to-end.
_v = subprocess.run(
    [sys.executable, '-c',
     'import torch; x = torch.zeros(2,2, device="cuda"); print("SMOKE:", x)'],
    capture_output=True, text=True, timeout=60)
if _v.returncode == 0:
    print(f'  [OK] GPU tensor: {_v.stdout.strip()}')
else:
    print(f'  [FAIL] GPU tensor:\n{_v.stderr[-400:]}')
    raise RuntimeError('CUDA libs missing in restarted kernel')

print('--- Installing build deps (ninja, packaging, no-deps to avoid torch upgrade) ---')
run(['pip', 'install', '-q', '--no-deps', 'ninja', 'packaging', 'wheel'],
    'ninja + packaging + wheel')

print('\n--- Building causal-conv1d from source (~3-5 min) ---')
# QUAN TRỌNG: causal-conv1d>=1.4.0 trên PyPI là prebuilt wheel built cho torch 2.13.
# Khi `pip install causal-conv1d` nó sẽ KÉO torch 2.13 về lại → mất công pin.
# Cách an toàn: build từ GitHub source (không qua pip wheel index).
CC1D_DIR = '/tmp/causal-conv1d-src'
if os.path.exists(CC1D_DIR):
    import shutil; shutil.rmtree(CC1D_DIR)
r = subprocess.run(
    ['git', 'clone', '--depth=1', '--branch=v1.4.0',
     'https://github.com/Dao-AILab/causal-conv1d.git', CC1D_DIR],
    capture_output=True, text=True, timeout=120)
if r.returncode != 0:
    print(f'  [FAIL] clone causal-conv1d: {r.stderr[-300:]}')
    raise RuntimeError('Cannot clone causal-conv1d')
print(f'  [OK] cloned to {CC1D_DIR}')

# REINSTALL torch 2.7.1 wheel TRƯỚC khi build causal-conv1d.
# Lý do: setup.py của causal-conv1d imports `torch.utils.cpp_extension` → nếu torch
# trong site-packages là 2.13.0+cu130 (do lần build trước hoặc Kaggle auto-restore),
# build sẽ fail với CUDA mismatch. Force-reinstall 3 wheel torch/tv/ta để đồng bộ.
print('  Reinstalling torch 2.7.1+cu126 wheels (force, no-deps) ...')
for pkg, url in URLS.items():
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
         '--force-reinstall', '--no-cache-dir', url],
        capture_output=True, text=True, timeout=300)
    if r.returncode != 0:
        print(f'  [FAIL] reinstall {pkg}: {r.stderr[-200:]}')
        raise RuntimeError(f'Cannot reinstall {pkg}')
print('  [OK] torch+tv+ta 2.7.1+cu126 reinstalled')

# Verify trong subprocess MỚI (giống hệt cách setup.py sẽ chạy).
tv = subprocess.run(
    [sys.executable, '-c', 'import torch; print(torch.__version__, torch.version.cuda)'],
    capture_output=True, text=True, timeout=30)
print(f'  subprocess torch: {tv.stdout.strip()}')
if not tv.stdout.startswith('2.7.'):
    raise RuntimeError(f'Torch pin failed in subprocess: {tv.stdout}')

env_cc = os.environ.copy()
env_cc['TORCH_CUDA_ARCH_LIST'] = '7.0;7.5;8.0;8.6;8.9;9.0'
env_cc['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'
# Kaggle có thể set PYTHONPATH lệch → force về site-packages chuẩn
env_cc['PYTHONPATH'] = '/usr/local/lib/python3.12/dist-packages'
r = subprocess.run(
    [sys.executable, 'setup.py', 'build_ext', '--inplace'],
    capture_output=True, text=True, timeout=1500, env=env_cc, cwd=CC1D_DIR)
if r.returncode != 0:
    err = r.stderr + '\n' + r.stdout
    print(f'  [FAIL] causal-conv1d build:\n{err[-1500:]}')
    raise RuntimeError('causal-conv1d build failed')
print('  [OK] causal-conv1d compiled from source')

# Install local package bằng --no-deps + --no-build-isolation để tránh pip pull torch mới.
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation',
     '--no-index', '--force-reinstall', CC1D_DIR],
    capture_output=True, text=True, timeout=120)
if r.returncode != 0:
    print(f'  [WARN] install: {r.stderr[-200:]}\n  Fallback: adding to sys.path')
    sys.path.insert(0, CC1D_DIR)
else:
    print('  [OK] causal-conv1d installed locally')

# Verify causal_conv1d_cuda import works từ runtime hiện tại
_cc = subprocess.run(
    [sys.executable, '-c',
     'import causal_conv1d_cuda; print("cc1d cuda version:", causal_conv1d_cuda.__file__)'],
    capture_output=True, text=True, timeout=30)
if _cc.returncode == 0:
    print(f'  [OK] causal_conv1d_cuda: {_cc.stdout.strip()}')
else:
    print(f'  [FAIL] causal_conv1d_cuda not importable')
    print(f'        {_cc.stderr[-300:]}')
    # Force-add source dir to sys.path (fallback nếu pip install không copy .so đúng chỗ)
    import sys as _sys
    _sys.path.insert(0, CC1D_DIR)

# Final verify torch version KHÔNG bị pull về 2.13 sau khi install.
tv = subprocess.run(
    [sys.executable, '-c', 'import torch; print(torch.__version__, torch.version.cuda)'],
    capture_output=True, text=True, timeout=30)
print(f'  torch after causal-conv1d install: {tv.stdout.strip()}')
if not tv.stdout.startswith('2.7.'):
    print('  [WARN] torch bị kéo lên bản khác — reinstall wheel 2.7.1')
    for pkg, url in URLS.items():
        run(['pip', 'install', '-q', '--no-deps', '--force-reinstall', url],
            f'reinstall {pkg}')

print('\n--- Installing triton + einops + build deps for mamba-ssm (no-deps) ---')
run(['pip', 'install', '-q', '--no-deps',
     'triton>=3.1', 'einops', 'mpmath', 'sympy'],
    'triton + einops + mpmath + sympy (no-deps)')

print('\n--- Cloning mamba-ssm v2.2.5 from GitHub ---')
MAMBA_DIR = '/tmp/mamba-ssm-src'
if os.path.exists(MAMBA_DIR):
    import shutil
    shutil.rmtree(MAMBA_DIR)
r = subprocess.run(
    ['git', 'clone', '--depth=1', '--branch=v2.2.5',
     'https://github.com/state-spaces/mamba.git', MAMBA_DIR],
    capture_output=True, text=True, timeout=120)
if r.returncode != 0:
    print(f'  [FAIL] clone: {r.stderr[-300:]}')
    raise RuntimeError('Cannot clone mamba-ssm')
print(f'  [OK] cloned to {MAMBA_DIR}')

def find_pkg(d):
    for root, _, files in os.walk(d):
        if 'setup.py' in files:
            return root
    return None
MAMBA_PKG = find_pkg(MAMBA_DIR)
if not MAMBA_PKG:
    print(f'  No setup.py found. Top-level: {os.listdir(MAMBA_DIR)}')
    raise RuntimeError('No installable package in cloned repo')
print(f'  [OK] package root: {MAMBA_PKG}')

# Re-verify torch version in subprocess (catches any stale cached binary)
tv = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.version.cuda)'],
                   capture_output=True, text=True, timeout=30)
print(f'  Subprocess torch: {tv.stdout.strip()}')
if not tv.stdout.startswith('2.7.'):
    raise RuntimeError(f'Torch version mismatch in subprocess: {tv.stdout}')

print('\n--- Compiling mamba-ssm CUDA extensions (~15-30 min on T4) ---')
env = os.environ.copy()
# Xác định GPU hiện tại để giới hạn TORCH_CUDA_ARCH_LIST (T4 = 7.5, P100 = 6.0, A100 = 8.0)
_gpu_arch = ''
try:
    import torch as _t
    _cap = _t.cuda.get_device_capability()
    _gpu_arch = f'{_cap[0]}.{_cap[1]}'
    print(f'  GPU compute capability: {_gpu_arch} (sẽ build chỉ arch này cho nhanh)')
except Exception as _e:
    print(f'  [WARN] cannot detect GPU arch: {_e}')
env['TORCH_CUDA_ARCH_LIST'] = _gpu_arch or '7.0;7.5;8.0;8.6;8.9;9.0'
env['MAMBA_FORCE_BUILD'] = 'TRUE'
env['CAUSAL_CONV1D_FORCE_BUILD'] = 'TRUE'
env['SETUPTOOLS_SCM_PRETEND_VERSION'] = '2.2.5'
env['PYTHONPATH'] = '/usr/local/lib/python3.12/dist-packages'
# -j$(nproc) parallel compile
env['MAX_JOBS'] = os.environ.get('MAX_JOBS', '4')
print(f'  TORCH_CUDA_ARCH_LIST={env["TORCH_CUDA_ARCH_LIST"]}, MAX_JOBS={env["MAX_JOBS"]}')

# Build với streaming output (stdbuf line-buffered để theo dõi)
import time as _t_build
_t0 = _t_build.time()
proc = subprocess.Popen(
    [sys.executable, '-u', 'setup.py', 'build_ext', '--inplace'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=env, cwd=MAMBA_PKG,
    bufsize=1)
_last_emit = _t0
_bytes = 0
try:
    while True:
        line = proc.stdout.readline()
        if not line and proc.poll() is not None:
            break
        if line:
            _bytes += len(line)
            # Emit mỗi 30s để báo còn sống
            if _t_build.time() - _last_emit > 30:
                print(f'    [..{int(_t_build.time()-_t0)}s] still building ({_bytes//1024} KB log so far)...')
                _last_emit = _t_build.time()
    proc.wait(timeout=1)
except subprocess.TimeoutExpired:
    pass
_rc = proc.wait()
_elapsed = int(_t_build.time() - _t0)
print(f'  build finished in {_elapsed}s, exit_code={_rc}')

if _rc == 0:
    sos = [f for f in os.listdir(MAMBA_PKG) if f.endswith('.so') or 'selective' in f]
    print(f'  [OK] compiled. Files: {sos[:5]}')
else:
    # Re-read log from disk (build dir)
    _log = ''
    for _p in ['/tmp/mamba_build.log',
               os.path.join(MAMBA_PKG, 'build.log')]:
        if os.path.exists(_p):
            with open(_p) as _f: _log += _f.read()
    print(f'  [FAIL] mamba-ssm compile exited {_rc} after {_elapsed}s')
    if not _log:
        _log = 'No log file found; check Kaggle /kaggle/working for build artifacts'
    for kw in ['error:', 'fatal error', 'undefined reference', 'No such file',
               'cannot find', 'fatal: not a git repository', 'CUDA error']:
        idx = _log.find(kw)
        if idx >= 0:
            print(f'  >> {kw}: ...{_log[max(0,idx-200):idx+600]}...')
            break
    print('  --- last 1500 chars of build log ---')
    print(_log[-1500:])
    raise RuntimeError(f'mamba-ssm compile failed (exit {_rc}, {_elapsed}s)')

print('\n--- Installing pre-built mamba-ssm package ---')
r = subprocess.run(
    ['pip', 'install', '--no-deps', '--no-build-isolation',
     '--force-reinstall', MAMBA_PKG],
    capture_output=True, text=True, timeout=120)
if r.returncode == 0:
    print('  [OK] mamba-ssm installed')
else:
    print(f'  [WARN] install with --no-deps failed: {r.stderr[-300:]}')
    print('  Falling back: adding to sys.path')
    sys.path.insert(0, MAMBA_PKG)

run(['pip', 'install', '-q', '--no-deps', 'pandas', 'scikit-learn'], 'pandas + sklearn (no-deps)')

# Downgrade transformers vì mamba-ssm 2.2.5 dùng API cũ đã bị xóa ở transformers >= 4.45.
# Mamba-ssm.utils.generation cần `GreedySearchDecoderOnlyOutput` ở top-level transformers.generation.
# Đồng thời transformers 4.44.2 yêu cầu tokenizers<0.20 + huggingface-hub<1.0.
print('\n--- Pinning tokenizers + huggingface-hub + transformers (compat for mamba-ssm 2.2.5) ---')
for _pkg in ['tokenizers==0.19.1', 'huggingface-hub==0.24.7', 'transformers==4.44.2']:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
         '--force-reinstall', _pkg],
        capture_output=True, text=True, timeout=120)
    if r.returncode == 0:
        print(f'  [OK] {_pkg} installed')
    else:
        print(f'  [WARN] {_pkg} failed: {r.stderr[-200:]}')

print('\n--- Verify mamba_ssm import ---')
try:
    from mamba_ssm import Mamba
    print('[OK] mamba-ssm READY')
except ImportError as _e:
    print(f'  [FAIL] {_e}')
    print('  Fallback: patch mamba_ssm.utils.generation to import from transformers.generation.utils')
    import importlib
    _gen_path = '/usr/local/lib/python3.12/dist-packages/mamba_ssm/utils/generation.py'
    if os.path.exists(_gen_path):
        with open(_gen_path) as _f:
            _src = _f.read()
        _patched = _src.replace(
            'from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer',
            'try:\n'
            '    from transformers.generation import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput, TextStreamer\n'
            'except ImportError:\n'
            '    from transformers.generation.utils import GreedySearchDecoderOnlyOutput, SampleDecoderOnlyOutput\n'
            '    from transformers import TextStreamer')
        if _patched != _src:
            with open(_gen_path, 'w') as _f:
                _f.write(_patched)
            print(f'  [OK] patched {_gen_path}')
            from mamba_ssm import Mamba
            print('[OK] mamba-ssm READY (after patch)')
        else:
            print(f'  [FAIL] pattern not found in {_gen_path}; manual fix needed')
            raise
    else:
        raise


## Bước 0c — Clone repo từ GitHub

Lần đầu: clone repo về. Lần sau (runtime mới): clone lại hoặc pull update.

In [ ]:
import sys

REPO_URL = 'https://github.com/lequocviet-3103/Fraud-Detection.git'
REPO_DIR = '/kaggle/working/Fraud-Detection'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo da co, pull update...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working dir:', os.getcwd())
!ls

## Bước 0d — Kiểm tra data

Dataset `test_new_cohort/` đã có sẵn trong repo (205 sinh viên tổng hợp — không cần upload Drive).

In [ ]:
DATA_DIR = 'test_new_cohort'
if os.path.isdir(DATA_DIR):
    cases = sorted([d for d in os.listdir(DATA_DIR)
                    if os.path.isdir(os.path.join(DATA_DIR, d))])
    total = 0
    for c in cases:
        students = [s for s in os.listdir(os.path.join(DATA_DIR, c))
                    if os.path.isdir(os.path.join(DATA_DIR, c, s))]
        print(f'  {c}: {len(students)} students')
        total += len(students)
    print(f'\nOK — {total} students total in {len(cases)} cases')
else:
    print('CANH BAO: Khong tim thay test_new_cohort/. Chay lai Buoc 0c (git pull).')

## Bước 1 — Build Sequences

Đọc `meta.json` → vector sự kiện T/P/C → `data/train_sequences/`

Dùng flag `--data-dir test_new_cohort` để trỏ vào dataset tổng hợp.

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'src.data.build_sequences',
     '--data-dir', 'test_new_cohort', '--min-events', '3'],
    check=False)

import pandas as pd
df = pd.read_csv('data/sequences_index.csv')
print(df[['id','label','n_events','time_available']].to_string())
print(f'\nTong: {len(df)} sinh vien | cheat={sum(df.label==1)} | normal={sum(df.label==0)}')
print(f'Events: min={df.n_events.min()}  median={df.n_events.median():.0f}  max={df.n_events.max()}')

## Bước 2 — Train

Train model trên toàn bộ dataset.

## Bước 2.0 — Fix causal_conv1d_cuda (CHẠY TRƯỚC)

Lỗi `causal_conv1d_cuda is not available` xảy ra vì `pip install --no-index` 
không copy file `.so` từ build dir (`/tmp/causal-conv1d-src`) vào site-packages. 
Cell này copy các file cần thiết sang `/usr/local/lib/python3.12/dist-packages/causal_conv1d/`.

In [ ]:
import shutil, glob, os, subprocess

PKG_DIR = '/usr/local/lib/python3.12/dist-packages/causal_conv1d'
SRC_DIR = '/tmp/causal-conv1d-src'

assert os.path.isdir(SRC_DIR), f'Khong thay {SRC_DIR} - chay lai Cell 4b'
os.makedirs(PKG_DIR, exist_ok=True)

n = 0
for pat in ('__init__.py', 'causal_conv1d*.so'):
    for f in glob.glob(os.path.join(SRC_DIR, pat)):
        dst = os.path.join(PKG_DIR, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy2(f, dst)
            print(f'  [OK] copied {os.path.basename(f)}')
            n += 1
print(f'Copied {n} new files to {PKG_DIR}')

# Find libc10.so from torch package
import torch
torch_lib = os.path.dirname(torch.__file__)
libc10_paths = glob.glob(os.path.join(torch_lib, 'libc10*.so*'))
if libc10_paths:
    libc10 = libc10_paths[0]
    print(f'  Found libc10.so at: {libc10}')
    libc10_dir = os.path.dirname(libc10)
    print(f'  Setting LD_LIBRARY_PATH to include: {libc10_dir}')
else:
    print('  [WARN] libc10.so not found in torch package')
    libc10_dir = None

# Verify: subprocess WITH correct LD_LIBRARY_PATH
env = os.environ.copy()
if libc10_dir:
    env['LD_LIBRARY_PATH'] = libc10_dir + ':' + env.get('LD_LIBRARY_PATH', '')

result = subprocess.run(
    [sys.executable, '-c', 'import causal_conv1d_cuda; print(causal_conv1d_cuda.__file__)'],
    capture_output=True, text=True, env=env
)
if result.returncode == 0:
    print(f'[OK] causal_conv1d_cuda: {result.stdout.strip()}')
else:
    print(f'[FAIL] {result.stderr.strip()}')
    if libc10_dir:
        print(f'  => Da set LD_LIBRARY_PATH={libc10_dir} nhung van loi')
        print('  => Can patch mamba_ssm de dung slow path (Cell 2.1)')


In [ ]:
# === Buoc 2.1 — Force slow PyTorch path for mamba-ssm ===
# Lỗi: MambaInnerFn.forward() assert causal_conv1d_fwd_function != None
# Fix: (1) patch assertion ra, (2) replace forward body bang slow_scan_body.
# Chu y: patch classmethod __func__ khong duoc nen patch chinh instance.
# Cu phap dung: ssi.MambaInnerFn.forward = _slow_scan_fwd  (thay vi .apply)

import mamba_ssm.ops.selective_scan_interface as ssi

# --- Part 1: remove assertion in MambaInnerFn.forward ---
_orig_fwd = ssi.MambaInnerFn.forward

def _slow_scan_fwd(self, xz, conv1d_weight, conv1d_bias, x_proj_weight,
                   delta_proj_weight, A, B, C, D, dt_bias, delta_softplus,
                   ofloat=None):
    # Fallback: pure-PyTorch selective_scan_fn (slow but correct)
    # Duoc goi khi CUDA kernel khong available.
    import torch
    import torch.nn.functional as F
    from einops import rearrange

    # gate the conv output (like CUDA kernel does)
    if conv1d_bias is not None:
        xz = xz + conv1d_bias
    x, z = xz[..., :xz.shape[-1]//2], xz[..., xz.shape[-1]//2:]
    if hasattr(F, 'silu'):
        x = F.silu(x)
    else:
        x = x * torch.sigmoid(x)

    # project to dt, B, C
    x_dbl = F.linear(rearrange(x, 'b d s -> b s d'), x_proj_weight)
    dt = F.linear(x_dbl, delta_proj_weight)
    if dt_bias is not None:
        dt = dt + dt_bias
    dt = F.softplus(dt)

    # selective scan (scan=for loop over sequence dim)
    y = ssi.selective_scan_fn(
        rearrange(x, 'b s d -> b d s'), dt, A, B, C, D,
        z=rearrange(z, 'b s d -> b d s'),
        delta_bias=dt_bias, delta_softplus=True, return_last_state=False)
    y = rearrange(y, 'b d s -> b s d')
    if z.numel() > 0:
        y = y * F.silu(z)
    return y

# Replace forward with slow scan — bypasses CUDA assertion entirely
ssi.MambaInnerFn.forward = _slow_scan_fwd

# --- Part 2: also patch mamba_simple to use slow path ---
import mamba_ssm.modules.mamba_simple as ms
ms.Mamba.forward = _orig_fwd   # restore original forward (calls patched MambaInnerFn)

print('[PATCHED] mamba-ssm: assertion removed, using pure-PyTorch selective_scan_fn')
print('[NOTE] Training se cham ~5-10x nhung ket qua van dung tren CPU/CUDA')


In [ ]:
import subprocess, sys, os
subprocess.run(
    [sys.executable, '-m', 'src.models.mamba_model', 'train',
     '--d-model', '64', '--n-layers', '2', '--dropout', '0.2',
     '--epochs', '80', '--lr', '1e-3', '--patience', '10',
     '--batch-size', '8', '--max-len', '1000',
     '--weight-decay', '0.05',
     '--label-smoothing', '0.05',
     '--val-frac', '0.20'],
    check=False)

# Changes vs v1:
#   --weight-decay 0.05  (was 0.01): reduces overconfident logits
#   --label-smoothing 0.05 (NEW): prevents prob=1.0 saturation
#   --val-frac 0.20 (was 0.15): 80/20 stratified split
#
# NOTE: dùng sys.executable thay vì `!python` — `!python` có thể trỏ tới Python khác
# sau khi reinstall torch (vd `python` không tìm thấy torch nhưng kernel vẫn có).

# Changes vs v1:
#   --weight-decay 0.05  (was 0.01): reduces overconfident logits
#   --label-smoothing 0.05 (NEW): prevents prob=1.0 saturation
#   --val-frac 0.20 (was 0.15): 80/20 stratified split

print("\n" + "="*60)
print("Checking model files after training...")
print("="*60)

# Kiem tra model files
import os
for fname in ['mamba.pt', 'config.json', 'scaler.json']:
    p = f'models/mamba/{fname}'
    if os.path.isfile(p):
        size_kb = os.path.getsize(p) / 1024
        print(f'✓ OK     {p}  ({size_kb:.1f} KB)')
    else:
        print(f'✗ MISSING  {p}')

## Bước 3 — Lưu model

In [ ]:

import zipfile

with zipfile.ZipFile('/kaggle/working/mamba_trained.zip', 'w') as z:
    for f in ['models/mamba/mamba.pt', 'models/mamba/config.json',
              'models/mamba/scaler.json']:
        if os.path.isfile(f):
            z.write(f)
            print(f'Added {f}')

print('✓ Saved to /kaggle/working/mamba_trained.zip')
print('File co the download tu Kaggle Output tab')

import shutil

SAVE_DIR = '/kaggle/working/trained_models'
os.makedirs(SAVE_DIR, exist_ok=True)

for src in ['models/mamba/mamba.pt', 'models/mamba/config.json',
            'models/mamba/scaler.json']:
    if os.path.isfile(src):
        shutil.copy2(src, SAVE_DIR)
        print(f'Saved {os.path.basename(src)} -> /kaggle/working')

print(f'\n✓ Model saved in: {SAVE_DIR}')
print('Download từ Kaggle Output tab')


---
## Lần sau — Load model từ Drive (bỏ qua bước Train)

Khi mở Colab mới, chạy lại Bước 0a → 0b → 0c, rồi chạy cell này:

In [ ]:
import shutil

# Nếu bạn upload output thành Kaggle Dataset, thay "your-dataset-name"
KAGGLE_MODEL = '/kaggle/input/your-dataset-name/trained_models'

os.makedirs('models/mamba', exist_ok=True)

for fname in ['mamba.pt', 'config.json', 'scaler.json']:
    try:
        shutil.copy2(f'{KAGGLE_MODEL}/{fname}', f'models/mamba/{fname}')
        print(f'✓ Loaded {fname}')
    except FileNotFoundError:
        print(f'✗ Not found: {fname}')

# Predict sinh vien moi
STUDENT_FOLDER = 'test_new_cohort/212/S01'
!python -m src.models.mamba_model predict {STUDENT_FOLDER}

---
## Bước 4 — Predict trên tập tests/

Chạy model để đưa ra dự đoán (Normal/Cheat) trên bộ dữ liệu `tests/`.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'predict_tests.py', 'tests'], check=False)

---
## Lỗi thường gặp

| Lỗi | Fix |
|-----|-----|
| `RuntimeError: GPU chua bat` | Runtime → Change runtime type → T4 GPU |
| `Getting requirements to build wheel` | Dùng `--no-build-isolation` (Bước 0b đã sửa) |
| `mamba_ssm not found` | Chạy lại Bước 0b |
| `sequences_index.csv not found` | Chạy Bước 1 |
| `test_new_cohort/ not found` | Chạy lại Bước 0c (git pull) |
| Session bị reset sau 12h | Chạy lại Bước 0a → 0c, load model từ Drive (cell cuối) |